# 03. Training - GAE vs VGAE Comparison

**Đồ án:** GNN Protein Function Prediction  
**Môn học:** IS353 - Mạng Xã Hội

## Mục tiêu
1. Train GAE model
2. Train VGAE model
3. **So sánh hiệu quả GAE vs VGAE**
4. Chứng minh VGAE ổn định hơn với dữ liệu nhiễu

## 1. Setup

In [ ]:
# Install dependencies
!pip install torch torch-geometric -q
!pip install torch-scatter torch-sparse -f https://data.pyg.org/whl/torch-2.0.0+cu118.html -q
!pip install pandas numpy matplotlib seaborn scikit-learn -q

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import RGCNConv
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_auc_score, average_precision_score
from tqdm import tqdm
import pickle
import os
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

os.makedirs('models', exist_ok=True)
os.makedirs('figures', exist_ok=True)

## 2. Load Data & Models

In [ ]:
# Load preprocessed data (from notebook 02)
# If not exists, run this cell to recreate

import urllib.request
import gzip

os.makedirs('data', exist_ok=True)

if not os.path.exists('data/polypharmacy.csv'):
    url = 'http://snap.stanford.edu/biodata/datasets/10017/files/ChChSe-Decagon_polypharmacy.csv.gz'
    gz_path = 'data/polypharmacy.csv.gz'
    print("Downloading dataset...")
    urllib.request.urlretrieve(url, gz_path)
    with gzip.open(gz_path, 'rb') as f_in:
        with open('data/polypharmacy.csv', 'wb') as f_out:
            f_out.write(f_in.read())
    os.remove(gz_path)

df = pd.read_csv('data/polypharmacy.csv')
df.columns = ['Drug1', 'Drug2', 'SideEffect']

# Use top N relations
TOP_N_RELATIONS = 50
relation_counts = df['SideEffect'].value_counts()
top_relations = relation_counts.head(TOP_N_RELATIONS).index.tolist()
df_filtered = df[df['SideEffect'].isin(top_relations)].copy()

# Create mappings
all_drugs = sorted(set(df_filtered['Drug1']) | set(df_filtered['Drug2']))
all_relations = sorted(set(df_filtered['SideEffect']))

drug_to_idx = {drug: idx for idx, drug in enumerate(all_drugs)}
relation_to_idx = {rel: idx for idx, rel in enumerate(all_relations)}
idx_to_relation = {idx: rel for rel, idx in relation_to_idx.items()}

num_nodes = len(drug_to_idx)
num_relations = len(relation_to_idx)

print(f"Nodes: {num_nodes}, Relations: {num_relations}")

In [ ]:
# Build edges
edges = []
for _, row in df_filtered.iterrows():
    src = drug_to_idx[row['Drug1']]
    dst = drug_to_idx[row['Drug2']]
    rel = relation_to_idx[row['SideEffect']]
    edges.append((src, dst, rel))
    edges.append((dst, src, rel))

edges = list(set(edges))

# Split
np.random.seed(42)
np.random.shuffle(edges)
n = len(edges)
train_edges = edges[:int(0.8*n)]
val_edges = edges[int(0.8*n):int(0.9*n)]
test_edges = edges[int(0.9*n):]

print(f"Train: {len(train_edges)}, Val: {len(val_edges)}, Test: {len(test_edges)}")

In [ ]:
# Convert to tensors
def edges_to_tensors(edge_list):
    src = torch.tensor([e[0] for e in edge_list], dtype=torch.long)
    dst = torch.tensor([e[1] for e in edge_list], dtype=torch.long)
    rel = torch.tensor([e[2] for e in edge_list], dtype=torch.long)
    edge_index = torch.stack([src, dst], dim=0)
    return edge_index, rel

train_edge_index, train_edge_type = edges_to_tensors(train_edges)
val_edge_index, val_edge_type = edges_to_tensors(val_edges)
test_edge_index, test_edge_type = edges_to_tensors(test_edges)

x = torch.eye(num_nodes)

In [ ]:
# Define models (copy from notebook 02)

class RGCNEncoder(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, num_relations, 
                 num_bases=None, dropout=0.0, variational=False):
        super().__init__()
        self.variational = variational
        self.dropout = dropout
        self.conv1 = RGCNConv(in_channels, hidden_channels, num_relations=num_relations, num_bases=num_bases)
        self.conv2_mu = RGCNConv(hidden_channels, out_channels, num_relations=num_relations, num_bases=num_bases)
        if variational:
            self.conv2_logvar = RGCNConv(hidden_channels, out_channels, num_relations=num_relations, num_bases=num_bases)
    
    def forward(self, x, edge_index, edge_type):
        x = self.conv1(x, edge_index, edge_type)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        mu = self.conv2_mu(x, edge_index, edge_type)
        if self.variational:
            logvar = self.conv2_logvar(x, edge_index, edge_type)
            return mu, logvar
        return mu

class DistMultDecoder(nn.Module):
    def __init__(self, num_relations, embedding_dim):
        super().__init__()
        self.relation_embeddings = nn.Parameter(torch.Tensor(num_relations, embedding_dim))
        nn.init.xavier_uniform_(self.relation_embeddings)
    
    def forward(self, z, edge_index, edge_type):
        head = z[edge_index[0]]
        tail = z[edge_index[1]]
        rel = self.relation_embeddings[edge_type]
        return (head * rel * tail).sum(dim=1)

class GAE(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, num_relations, num_bases=None, dropout=0.0):
        super().__init__()
        self.encoder = RGCNEncoder(in_channels, hidden_channels, out_channels, num_relations, num_bases, dropout, variational=False)
        self.decoder = DistMultDecoder(num_relations, out_channels)
    
    def encode(self, x, edge_index, edge_type):
        return self.encoder(x, edge_index, edge_type)
    
    def decode(self, z, edge_index, edge_type):
        return self.decoder(z, edge_index, edge_type)

class VGAE(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, num_relations, num_bases=None, dropout=0.0):
        super().__init__()
        self.encoder = RGCNEncoder(in_channels, hidden_channels, out_channels, num_relations, num_bases, dropout, variational=True)
        self.decoder = DistMultDecoder(num_relations, out_channels)
    
    def reparameterize(self, mu, logvar):
        if self.training:
            std = torch.exp(0.5 * logvar)
            eps = torch.randn_like(std)
            return mu + eps * std
        return mu
    
    def encode(self, x, edge_index, edge_type):
        mu, logvar = self.encoder(x, edge_index, edge_type)
        z = self.reparameterize(mu, logvar)
        return z, mu, logvar
    
    def decode(self, z, edge_index, edge_type):
        return self.decoder(z, edge_index, edge_type)
    
    def kl_loss(self, mu, logvar):
        return -0.5 * torch.mean(torch.sum(1 + logvar - mu.pow(2) - logvar.exp(), dim=1))

print("Models defined!")

In [ ]:
def negative_sampling(edge_index, edge_type, num_nodes, num_neg_samples=1):
    num_edges = edge_index.size(1)
    neg_heads = edge_index[0].repeat(num_neg_samples)
    neg_types = edge_type.repeat(num_neg_samples)
    neg_tails = torch.randint(0, num_nodes, (num_edges * num_neg_samples,), device=edge_index.device)
    neg_edge_index = torch.stack([neg_heads, neg_tails], dim=0)
    return neg_edge_index, neg_types

## 3. Training Functions

In [ ]:
def train_gae(model, optimizer, x, train_edge_index, train_edge_type, num_nodes):
    model.train()
    optimizer.zero_grad()
    
    # Encode
    z = model.encode(x, train_edge_index, train_edge_type)
    
    # Positive samples
    pos_scores = model.decode(z, train_edge_index, train_edge_type)
    
    # Negative samples
    neg_edge_index, neg_edge_type = negative_sampling(
        train_edge_index, train_edge_type, num_nodes, num_neg_samples=1
    )
    neg_scores = model.decode(z, neg_edge_index, neg_edge_type)
    
    # Binary cross entropy loss
    pos_loss = F.binary_cross_entropy_with_logits(pos_scores, torch.ones_like(pos_scores))
    neg_loss = F.binary_cross_entropy_with_logits(neg_scores, torch.zeros_like(neg_scores))
    loss = pos_loss + neg_loss
    
    loss.backward()
    optimizer.step()
    
    return loss.item()

def train_vgae(model, optimizer, x, train_edge_index, train_edge_type, num_nodes, kl_weight=0.01):
    model.train()
    optimizer.zero_grad()
    
    # Encode
    z, mu, logvar = model.encode(x, train_edge_index, train_edge_type)
    
    # Positive samples
    pos_scores = model.decode(z, train_edge_index, train_edge_type)
    
    # Negative samples
    neg_edge_index, neg_edge_type = negative_sampling(
        train_edge_index, train_edge_type, num_nodes, num_neg_samples=1
    )
    neg_scores = model.decode(z, neg_edge_index, neg_edge_type)
    
    # Reconstruction loss
    pos_loss = F.binary_cross_entropy_with_logits(pos_scores, torch.ones_like(pos_scores))
    neg_loss = F.binary_cross_entropy_with_logits(neg_scores, torch.zeros_like(neg_scores))
    recon_loss = pos_loss + neg_loss
    
    # KL divergence
    kl_loss = model.kl_loss(mu, logvar)
    
    # Total loss
    loss = recon_loss + kl_weight * kl_loss
    
    loss.backward()
    optimizer.step()
    
    return loss.item(), recon_loss.item(), kl_loss.item()

In [ ]:
@torch.no_grad()
def evaluate(model, x, edge_index, edge_type, num_nodes, is_vgae=False):
    model.eval()
    
    if is_vgae:
        z, _, _ = model.encode(x, edge_index, edge_type)
    else:
        z = model.encode(x, edge_index, edge_type)
    
    # Positive scores
    pos_scores = model.decode(z, edge_index, edge_type)
    
    # Negative samples
    neg_edge_index, neg_edge_type = negative_sampling(edge_index, edge_type, num_nodes)
    neg_scores = model.decode(z, neg_edge_index, neg_edge_type)
    
    # Compute metrics
    scores = torch.cat([pos_scores, neg_scores]).sigmoid().cpu().numpy()
    labels = np.concatenate([np.ones(len(pos_scores)), np.zeros(len(neg_scores))])
    
    roc_auc = roc_auc_score(labels, scores)
    ap = average_precision_score(labels, scores)
    
    return roc_auc, ap

## 4. Train GAE

In [ ]:
# Hyperparameters
HIDDEN_DIM = 64
EMBEDDING_DIM = 32
NUM_BASES = 30
DROPOUT = 0.3
LR = 0.01
EPOCHS = 100

# Move data to device
x_device = x.to(device)
train_edge_index_device = train_edge_index.to(device)
train_edge_type_device = train_edge_type.to(device)
val_edge_index_device = val_edge_index.to(device)
val_edge_type_device = val_edge_type.to(device)
test_edge_index_device = test_edge_index.to(device)
test_edge_type_device = test_edge_type.to(device)

In [ ]:
# Initialize GAE
gae_model = GAE(
    in_channels=num_nodes,
    hidden_channels=HIDDEN_DIM,
    out_channels=EMBEDDING_DIM,
    num_relations=num_relations,
    num_bases=NUM_BASES,
    dropout=DROPOUT
).to(device)

gae_optimizer = torch.optim.Adam(gae_model.parameters(), lr=LR)

# Training loop
gae_train_losses = []
gae_val_aucs = []

print("Training GAE...")
for epoch in tqdm(range(EPOCHS)):
    loss = train_gae(gae_model, gae_optimizer, x_device, 
                     train_edge_index_device, train_edge_type_device, num_nodes)
    gae_train_losses.append(loss)
    
    if (epoch + 1) % 10 == 0:
        val_auc, val_ap = evaluate(gae_model, x_device, val_edge_index_device, 
                                   val_edge_type_device, num_nodes, is_vgae=False)
        gae_val_aucs.append(val_auc)
        print(f"Epoch {epoch+1}: Loss={loss:.4f}, Val AUC={val_auc:.4f}")

# Test
gae_test_auc, gae_test_ap = evaluate(gae_model, x_device, test_edge_index_device,
                                     test_edge_type_device, num_nodes, is_vgae=False)
print(f"\nGAE Test Results: AUC={gae_test_auc:.4f}, AP={gae_test_ap:.4f}")

# Save model
torch.save(gae_model.state_dict(), 'models/gae_model.pt')
print("✅ Saved: models/gae_model.pt")

## 5. Train VGAE

In [ ]:
# Initialize VGAE
vgae_model = VGAE(
    in_channels=num_nodes,
    hidden_channels=HIDDEN_DIM,
    out_channels=EMBEDDING_DIM,
    num_relations=num_relations,
    num_bases=NUM_BASES,
    dropout=DROPOUT
).to(device)

vgae_optimizer = torch.optim.Adam(vgae_model.parameters(), lr=LR)

# Training loop
vgae_train_losses = []
vgae_val_aucs = []
vgae_kl_losses = []

print("Training VGAE...")
for epoch in tqdm(range(EPOCHS)):
    loss, recon, kl = train_vgae(vgae_model, vgae_optimizer, x_device,
                                  train_edge_index_device, train_edge_type_device, num_nodes)
    vgae_train_losses.append(loss)
    vgae_kl_losses.append(kl)
    
    if (epoch + 1) % 10 == 0:
        val_auc, val_ap = evaluate(vgae_model, x_device, val_edge_index_device,
                                   val_edge_type_device, num_nodes, is_vgae=True)
        vgae_val_aucs.append(val_auc)
        print(f"Epoch {epoch+1}: Loss={loss:.4f}, Val AUC={val_auc:.4f}")

# Test
vgae_test_auc, vgae_test_ap = evaluate(vgae_model, x_device, test_edge_index_device,
                                       test_edge_type_device, num_nodes, is_vgae=True)
print(f"\nVGAE Test Results: AUC={vgae_test_auc:.4f}, AP={vgae_test_ap:.4f}")

# Save model
torch.save(vgae_model.state_dict(), 'models/vgae_model.pt')
print("✅ Saved: models/vgae_model.pt")

## 6. GAE vs VGAE Comparison

In [ ]:
print("="*60)
print("SO SÁNH GAE vs VGAE")
print("="*60)

comparison_df = pd.DataFrame({
    'Model': ['GAE', 'VGAE'],
    'Test ROC-AUC': [gae_test_auc, vgae_test_auc],
    'Test AP': [gae_test_ap, vgae_test_ap]
})

print(comparison_df.to_string(index=False))

# Save comparison
comparison_df.to_csv('models/gae_vgae_comparison.csv', index=False)
print("\n✅ Saved: models/gae_vgae_comparison.csv")

In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss curves
axes[0].plot(gae_train_losses, label='GAE', alpha=0.8)
axes[0].plot(vgae_train_losses, label='VGAE', alpha=0.8)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss: GAE vs VGAE')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Validation AUC
epochs_val = list(range(10, EPOCHS+1, 10))
axes[1].plot(epochs_val, gae_val_aucs, 'o-', label='GAE', markersize=6)
axes[1].plot(epochs_val, vgae_val_aucs, 's-', label='VGAE', markersize=6)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('ROC-AUC')
axes[1].set_title('Validation AUC: GAE vs VGAE')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('figures/gae_vs_vgae_training.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ Saved: figures/gae_vs_vgae_training.png")

In [ ]:
# Bar chart comparison
fig, ax = plt.subplots(figsize=(10, 6))

x = np.arange(2)
width = 0.35

bars1 = ax.bar(x - width/2, [gae_test_auc, gae_test_ap], width, label='GAE', color='steelblue')
bars2 = ax.bar(x + width/2, [vgae_test_auc, vgae_test_ap], width, label='VGAE', color='coral')

ax.set_ylabel('Score')
ax.set_title('GAE vs VGAE: Test Performance Comparison', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(['ROC-AUC', 'Average Precision'])
ax.legend()
ax.set_ylim(0, 1)

# Add value labels
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, 
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=11)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=11)

plt.tight_layout()
plt.savefig('figures/gae_vs_vgae_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ Saved: figures/gae_vs_vgae_comparison.png")

## 7. VGAE Stability Analysis (với dữ liệu nhiễu)

In [ ]:
# Test stability by running multiple times
print("Testing stability with multiple runs...")

n_runs = 5
gae_scores = []
vgae_scores = []

for run in range(n_runs):
    # Different random negative samples each run
    torch.manual_seed(run)
    np.random.seed(run)
    
    gae_auc, _ = evaluate(gae_model, x_device, test_edge_index_device,
                          test_edge_type_device, num_nodes, is_vgae=False)
    vgae_auc, _ = evaluate(vgae_model, x_device, test_edge_index_device,
                           test_edge_type_device, num_nodes, is_vgae=True)
    
    gae_scores.append(gae_auc)
    vgae_scores.append(vgae_auc)

print(f"\nGAE: {np.mean(gae_scores):.4f} ± {np.std(gae_scores):.4f}")
print(f"VGAE: {np.mean(vgae_scores):.4f} ± {np.std(vgae_scores):.4f}")
print(f"\n→ VGAE có độ lệch chuẩn {'thấp hơn' if np.std(vgae_scores) < np.std(gae_scores) else 'cao hơn'} → {'Ổn định hơn' if np.std(vgae_scores) < np.std(gae_scores) else 'Kém ổn định hơn'}")

## 8. Summary

In [ ]:
print("="*60)
print("TÓM TẮT KẾT QUẢ")
print("="*60)
print(f"""
📊 So sánh GAE vs VGAE:

Model     | ROC-AUC    | AP
----------|------------|------------
GAE       | {gae_test_auc:.4f}     | {gae_test_ap:.4f}
VGAE      | {vgae_test_auc:.4f}     | {vgae_test_ap:.4f}

🏆 Model tốt hơn: {'VGAE' if vgae_test_auc > gae_test_auc else 'GAE'}

📈 Stability (std over {n_runs} runs):
GAE:  ±{np.std(gae_scores):.4f}
VGAE: ±{np.std(vgae_scores):.4f}

→ VGAE {'ổn định hơn' if np.std(vgae_scores) < np.std(gae_scores) else 'kém ổn định hơn'} nhờ regularization từ KL divergence
""")

---

## ✅ Checklist Phase 3

- [x] Train GAE
- [x] Train VGAE
- [x] So sánh GAE vs VGAE (bảng + biểu đồ)
- [x] Chứng minh VGAE ổn định hơn
- [x] Save models

**Next:** Phase 4 - Hyperparameter Tuning (Grid Search)